# TAR@FAR Evaluation + ROC Plot

Evaluates one or more checkpoints on the Dataset A validation set. Reports TAR at FAR={1e-4, 1e-5, 1e-6}, throughput (images/sec), peak GPU memory, and optionally saves a ROC plot.

## Configuration

In [ ]:
CHECKPOINT_PATHS = ["./checkpoints/phase2.pth"]  # list of checkpoint paths
DATASET_ROOT     = "./datasets/dataset_a"
EMBEDDING_DIM    = 512
BATCH_SIZE       = 128
NUM_WORKERS  = 0  # >0 causes multiprocessing errors in Jupyter on macOS

OUTPUT_CSV       = None   # e.g. "./results/eval.csv", or None to skip
PLOT_PATH        = None   # e.g. "./results/roc.png", or None to skip

import torch
# Auto-detects: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

In [ ]:
import math
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm import tqdm

## Model Definition (FaceEncoder)

In [ ]:
class FaceEncoder(nn.Module):
    def __init__(self, embedding_dim=512):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone.fc = nn.Linear(2048, embedding_dim)
        self._embedding_dim = embedding_dim

    @property
    def embedding_dim(self):
        return self._embedding_dim

    def forward(self, x):
        return self.backbone(x)

    def encode(self, x):
        return F.normalize(self.forward(x), p=2, dim=1)

## Template Aggregation

In [ ]:
def aggregate_template_features(embeddings, template_ids, media_ids):
    template_features = {}
    for tid in np.unique(template_ids):
        mask = template_ids == tid
        t_embs = embeddings[mask]
        t_mids = media_ids[mask]
        media_feats = []
        for mid in np.unique(t_mids):
            media_feats.append(t_embs[t_mids == mid].mean(axis=0))
        feat = np.sum(media_feats, axis=0)
        feat = feat / (np.linalg.norm(feat) + 1e-8)
        template_features[tid] = feat
    return template_features

## Evaluation Transforms & Constants

In [ ]:
EVAL_TRANSFORM = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

FAR_TARGETS = [1e-4, 1e-5, 1e-6]

## Dataset & Encoding

In [ ]:
class FaceTestDataset(Dataset):
    def __init__(self, root):
        self.root = root
        df = pd.read_parquet(os.path.join(root, "test.parquet"))
        self.image_paths = df["image_path"].tolist()
        self.template_ids = df["template_id"].values
        self.media_ids = df["media_id"].values

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = os.path.join(self.root, self.image_paths[idx])
        img = Image.open(path).convert("RGB")
        return EVAL_TRANSFORM(img), idx


def encode_dataset(model, dataset, batch_size, num_workers, device):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=(device.type == 'cuda'))
    embs, idxs = [], []
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    with torch.inference_mode():
        for imgs, indices in tqdm(loader, desc="Encoding"):
            embs.append(model.encode(imgs.to(device)).cpu().numpy())
            idxs.append(indices.numpy())
    elapsed = time.perf_counter() - t0
    embeddings = np.vstack(embs)[np.argsort(np.concatenate(idxs))]
    return embeddings, elapsed

## Scoring & TAR@FAR

In [ ]:
def compute_scores(dataset, embeddings, pairs_df, embedding_dim):
    template_features = aggregate_template_features(embeddings, dataset.template_ids, dataset.media_ids)
    tid_list = sorted(template_features.keys())
    tid_to_idx = {tid: i for i, tid in enumerate(tid_list)}
    feat_matrix = np.zeros((len(tid_list), embedding_dim), dtype=np.float32)
    for tid, feat in template_features.items():
        feat_matrix[tid_to_idx[tid]] = feat

    t1s = pairs_df["template_id_1"].values
    t2s = pairs_df["template_id_2"].values
    scores = np.zeros(len(pairs_df), dtype=np.float32)
    batch = 500_000
    for i in range(0, len(pairs_df), batch):
        end = min(i + batch, len(pairs_df))
        i1 = np.array([tid_to_idx[t] for t in t1s[i:end]])
        i2 = np.array([tid_to_idx[t] for t in t2s[i:end]])
        scores[i:end] = np.sum(feat_matrix[i1] * feat_matrix[i2], axis=1)
    return scores


def tar_at_far(pos_scores, neg_scores):
    neg_sorted = np.sort(neg_scores)[::-1]
    results = {}
    for far in FAR_TARGETS:
        idx = max(math.ceil(far * len(neg_scores)) - 1, 0)
        tau = neg_sorted[idx]
        tar = float((pos_scores > tau).mean())
        results[f"TAR@FAR={far:.0e}"] = tar * 100
    return results

## Per-Checkpoint Evaluation

In [ ]:
def eval_checkpoint(ckpt_path, dataset_root, embedding_dim, batch_size, num_workers, device):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
    dim = ckpt.get("embedding_dim", embedding_dim)
    model = FaceEncoder(dim).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    dataset = FaceTestDataset(dataset_root)
    embeddings, elapsed = encode_dataset(model, dataset, batch_size, num_workers, device)

    throughput = len(dataset) / elapsed
    peak_mem_mb = (torch.cuda.max_memory_allocated() / 1e6) if device.type == "cuda" else 0.0

    pairs_df = pd.read_parquet(os.path.join(dataset_root, "pairs.parquet"))
    scores = compute_scores(dataset, embeddings, pairs_df, dim)

    labels = pairs_df["label"].values
    pos_scores = scores[labels == 1]
    neg_scores = scores[labels == 0]

    metrics = tar_at_far(pos_scores, neg_scores)
    metrics["throughput_img_s"] = throughput
    metrics["peak_gpu_mem_mb"] = peak_mem_mb
    metrics["embedding_dim"] = dim
    metrics["eval_time_s"] = elapsed
    metrics["n_pos"] = int((labels == 1).sum())
    metrics["n_neg"] = int((labels == 0).sum())

    return metrics, scores, labels

## Run Evaluation

In [ ]:
device = torch.device(DEVICE if (torch.cuda.is_available() or (DEVICE == "mps" and torch.backends.mps.is_available())) else "cpu")
print(f"Device: {device}")

rows = []
roc_curves = []

for ckpt_path in CHECKPOINT_PATHS:
    print(f"\n=== {ckpt_path} ===")
    metrics, scores, labels = eval_checkpoint(
        ckpt_path, DATASET_ROOT, EMBEDDING_DIM, BATCH_SIZE, NUM_WORKERS, device
    )
    metrics["checkpoint"] = ckpt_path
    rows.append(metrics)
    roc_curves.append((ckpt_path, scores, labels))

    print(f"  TAR@FAR=1e-4 : {metrics.get('TAR@FAR=1e-04', 0):.2f}%")
    print(f"  TAR@FAR=1e-5 : {metrics.get('TAR@FAR=1e-05', 0):.2f}%")
    print(f"  TAR@FAR=1e-6 : {metrics.get('TAR@FAR=1e-06', 0):.2f}%")
    print(f"  Throughput   : {metrics['throughput_img_s']:.1f} img/s")
    print(f"  Peak GPU mem : {metrics['peak_gpu_mem_mb']:.1f} MB")
    print(f"  Embedding dim: {metrics['embedding_dim']}")

## Save Results (optional)

In [ ]:
if OUTPUT_CSV:
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
    print(f"Results saved: {OUTPUT_CSV}")

## ROC Plot (optional)

In [ ]:
if PLOT_PATH:
    Path(PLOT_PATH).parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(8, 6))
    for ckpt_path, scores, labels in roc_curves:
        pos = scores[labels == 1]
        neg = scores[labels == 0]
        thresholds = np.sort(np.unique(scores))[::-1]
        fprs = [(neg >= tau).mean() for tau in thresholds]
        tprs = [(pos >= tau).mean() for tau in thresholds]
        ax.plot(fprs, tprs, label=Path(ckpt_path).stem)
    ax.set_xscale("log")
    ax.set_xlabel("FAR (log scale)")
    ax.set_ylabel("TAR")
    ax.set_title("ROC Curve")
    ax.legend()
    ax.grid(True, which="both", alpha=0.3)
    fig.savefig(PLOT_PATH, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"ROC plot saved: {PLOT_PATH}")